In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.spatial.transform import Rotation as Rotation

# проверить наблюдаемость фильтра 

In [ ]:



# ============================================================
#                    НАСТРОЙКИ ЭКСПЕРИМЕНТА
# ============================================================

np.random.seed(42)

# Время моделирования
dt = 1.0                    # шаг, с
T = 600.0                   # 15 минут
t = np.arange(0, T + dt, dt)
N = len(t)

# ------------------------------------------------------------
# Параметры участка сближения
#
# Камера ОР наблюдает ОО.
# Начальная дальность примерно 100 м.
# Конечная дальность примерно 20 м.
# ------------------------------------------------------------

r0 = np.array([
    80.0,       # x, м
    -50.0,      # y, м
    30.0        # z, м
])

rf = np.array([
    16.0,       # x, м
    -10.0,      # y, м
    6.0         # z, м
])

# Начальная дальность
rho0 = np.linalg.norm(r0)

# Конечная дальность
rhof = np.linalg.norm(rf)

print("Начальная дальность:", round(rho0, 2), "м")
print("Конечная дальность:", round(rhof, 2), "м")


# ============================================================
#          1. ИСТИННОЕ ПОСТУПАТЕЛЬНОЕ ДВИЖЕНИЕ
# ============================================================

# Базовая постоянная скорость сближения
v_base = (rf - r0) / T

# Создаем небольшие плавные изменения скорости,
# чтобы движение не было идеально линейным.
ax = 0.00008 * np.sin(2 * np.pi * t / 300)
ay = 0.00005 * np.cos(2 * np.pi * t / 250)
az = 0.00003 * np.sin(2 * np.pi * t / 400)

a_true = np.column_stack((ax, ay, az))

# Истинная скорость
v_true = np.zeros((N, 3))
v_true[0] = v_base

# Истинное положение
r_true = np.zeros((N, 3))
r_true[0] = r0

for k in range(N - 1):

    v_true[k + 1] = (
        v_true[k]
        +
        a_true[k] * dt
    )

    r_true[k + 1] = (
        r_true[k]
        +
        v_true[k] * dt
        +
        0.5 * a_true[k] * dt**2
    )


# Корректируем траекторию,
# чтобы финальная точка была около заданной rf.
correction = rf - r_true[-1]

for k in range(N):

    alpha = t[k] / T

    r_true[k] += alpha * correction


# Пересчет истинной скорости
v_true = np.gradient(r_true, dt, axis=0)


# ============================================================
#          2. ДАННЫЕ, ПОЛУЧЕННЫЕ ПРЕДЛОЖЕННЫМ МЕТОДОМ
# ============================================================

# Имитируем ошибку определения положения по изображению.
sigma_position = 0.7        # м

r_method = (r_true + np.random.normal(0, sigma_position, (N, 3)))


# Скорость методом определяется
# по последовательности оценок положения.

v_method = np.gradient(r_method, dt, axis=0)


# ============================================================
#          3. ФИЛЬТР КАЛМАНА
#       СОСТОЯНИЕ: [x y z vx vy vz]
# ============================================================

# Вектор состояния
x_kf = np.zeros((N, 6))

# Начальная оценка
x_kf[0, 0:3] = r_method[0]

x_kf[0, 3:6] = v_method[0]


# Матрица перехода состояния

F = np.block([[np.eye(3), np.eye(3) * dt],
              [np.zeros((3, 3)), np.eye(3)]])

# Матрица измерений.
# Измеряется только положение.

H = np.block([np.eye(3), np.zeros((3, 3))])

# Шум процесса
sigma_acc = 0.01
Q = np.block([[np.eye(3) * (0.5 * dt**2 * sigma_acc)**2, np.eye(3) * (0.5 * dt**3 * sigma_acc**2)],
              [np.eye(3) * (0.5 * dt**3 * sigma_acc**2), np.eye(3) * (dt * sigma_acc)**2]])

# Шум измерений
R = np.eye(3) * sigma_position**2
# Начальная ковариация

P = np.diag([

    4.0,
    4.0,
    4.0,

    0.05,
    0.05,
    0.05

])


I6 = np.eye(6)


for k in range(N - 1):

    # --------------------------------------------------------
    # ПРОГНОЗ
    # --------------------------------------------------------

    x_pred = F @ x_kf[k]
    P_pred = (F @ P @ F.T + Q)

    # --------------------------------------------------------
    # ИЗМЕРЕНИЕ
    # --------------------------------------------------------
    z = r_method[k + 1]
    # --------------------------------------------------------
    # НЕВЯЗКА
    # --------------------------------------------------------
    innovation = (z-H @ x_pred)

    # --------------------------------------------------------
    # КОВАРИАЦИЯ НЕВЯЗКИ
    # --------------------------------------------------------

    S = (H@ P_pred@ H.T+R)

    # --------------------------------------------------------
    # УСИЛЕНИЕ КАЛМАНА
    # --------------------------------------------------------

    K = (P_pred@ H.T@ np.linalg.inv(S))

    # --------------------------------------------------------
    # КОРРЕКЦИЯ СОСТОЯНИЯ
    # --------------------------------------------------------

    x_kf[k + 1] = (x_pred+K @ innovation)

    # --------------------------------------------------------
    # КОРРЕКЦИЯ КОВАРИАЦИИ
    # --------------------------------------------------------
    P = (I6 -K @ H) @ P_pred

r_kf = x_kf[:, 0:3]
v_kf = x_kf[:, 3:6]


# ============================================================
#          4. ИСТИННОЕ ВРАЩАТЕЛЬНОЕ ДВИЖЕНИЕ
# ============================================================

# Истинная угловая скорость.
# Значения в градусах в секунду.

omega_x_true_deg = (0.15+1+0.015 * np.sin(2 * np.pi * t / 300))
omega_y_true_deg = (-0.1-0.2+0.010 * np.cos(2 * np.pi * t / 400))
omega_z_true_deg = (0.2 + 0.7+0.012 * np.sin(2 * np.pi * t / 250))


omega_true_deg = np.column_stack((

    omega_x_true_deg,

    omega_y_true_deg,

    omega_z_true_deg

))


# Перевод в радианы

omega_true = np.deg2rad(
    omega_true_deg
)


# ============================================================
#      5. ИНТЕГРИРОВАНИЕ ИСТИННОЙ ОРИЕНТАЦИИ
# ============================================================

# q_true будет храниться в формате scipy:
#
# [qx, qy, qz, qw]

q_true = np.zeros((N, 4))

q_true[0] = np.array([

    0.0,
    0.0,
    0.0,
    1.0

])


for k in range(N - 1):

    # Малый поворот за dt

    rotation_increment = Rotation.from_rotvec(

        omega_true[k] * dt

    )


    current_rotation = Rotation.from_quat(

        q_true[k]

    )


    next_rotation = (

        current_rotation
        *
        rotation_increment

    )


    q_true[k + 1] = (

        next_rotation.as_quat()

    )


# ============================================================
#        6. ОРИЕНТАЦИЯ, ПОЛУЧЕННАЯ МЕТОДОМ
# ============================================================

# Ошибка определения ориентации камерой.
# 0.8 градуса.

sigma_orientation_deg = 0.8

q_method = np.zeros((N, 4))


for k in range(N):

    noise_rotation = Rotation.from_rotvec(

        np.deg2rad(

            np.random.normal(
                0,
                sigma_orientation_deg,
                3
            )

        )

    )


    true_rotation = Rotation.from_quat(

        q_true[k]

    )


    measured_rotation = (

        noise_rotation
        *
        true_rotation

    )


    q_method[k] = (

        measured_rotation.as_quat()

    )


# ============================================================
#     7. УГЛОВАЯ СКОРОСТЬ, ОЦЕНЕННАЯ МЕТОДОМ
# ============================================================

omega_method = np.zeros((N, 3))


for k in range(N - 1):

    R_k = Rotation.from_quat(
        q_method[k]
    )


    R_k1 = Rotation.from_quat(
        q_method[k + 1]
    )


    # Относительный поворот
    delta_rotation = (

        R_k.inv()
        *
        R_k1

    )


    # Вектор поворота
    delta_theta = (

        delta_rotation.as_rotvec()

    )


    # Оценка угловой скорости
    omega_method[k] = (

        delta_theta / dt

    )


# Последнее значение

omega_method[-1] = omega_method[-2]


# ============================================================
#       8. MEKF ДЛЯ ОРИЕНТАЦИИ И УГЛОВОЙ СКОРОСТИ
#
# Состояние ошибки:
#
# delta_x =
#
# [delta_theta_x
#  delta_theta_y
#  delta_theta_z
#  delta_omega_x
#  delta_omega_y
#  delta_omega_z]
# ============================================================


# Номинальный кватернион

q_mekf = np.zeros((N, 4))

q_mekf[0] = q_method[0]


# Номинальная угловая скорость

omega_mekf = np.zeros((N, 3))

omega_mekf[0] = omega_method[0]


# Ковариационная матрица

P_att = np.diag([

    np.deg2rad(5.0)**2,
    np.deg2rad(5.0)**2,
    np.deg2rad(5.0)**2,

    np.deg2rad(0.2)**2,
    np.deg2rad(0.2)**2,
    np.deg2rad(0.2)**2

])


# Шум модели

sigma_theta_process = np.deg2rad(0.01)

sigma_omega_process = np.deg2rad(0.005)


Q_att = np.diag([

    sigma_theta_process**2,
    sigma_theta_process**2,
    sigma_theta_process**2,

    sigma_omega_process**2,
    sigma_omega_process**2,
    sigma_omega_process**2

])


# Ошибка измерения ориентации

sigma_attitude = np.deg2rad(
    sigma_orientation_deg
)


R_att = np.eye(3) * sigma_attitude**2


# Матрица измерений

H_att = np.hstack([

    np.eye(3),

    np.zeros((3, 3))

])


I_att = np.eye(6)


for k in range(N - 1):

    # --------------------------------------------------------
    # 1. ПРОГНОЗ ОРИЕНТАЦИИ
    # --------------------------------------------------------

    q_rotation = Rotation.from_quat(

        q_mekf[k]

    )


    rotation_increment = Rotation.from_rotvec(

        omega_mekf[k] * dt

    )


    q_pred_rotation = (

        q_rotation
        *
        rotation_increment

    )


    q_pred = (

        q_pred_rotation.as_quat()

    )


    # --------------------------------------------------------
    # 2. ПРОГНОЗ УГЛОВОЙ СКОРОСТИ
    #
    # omega(k+1) = omega(k)
    # --------------------------------------------------------

    omega_pred = omega_mekf[k].copy()


    # --------------------------------------------------------
    # 3. МАТРИЦА ПЕРЕХОДА ОШИБКИ
    # --------------------------------------------------------

    omega = omega_pred


    omega_cross = np.array([

        [0.0,       -omega[2],  omega[1]],

        [omega[2],  0.0,       -omega[0]],

        [-omega[1], omega[0],   0.0]

    ])


    Phi = np.block([

        [

            np.eye(3)
            -
            omega_cross * dt,

            np.eye(3) * dt

        ],

        [

            np.zeros((3, 3)),

            np.eye(3)

        ]

    ])


    # --------------------------------------------------------
    # 4. ПРОГНОЗ КОВАРИАЦИИ
    # --------------------------------------------------------

    P_pred = (

        Phi
        @ P_att
        @ Phi.T
        +
        Q_att

    )


    # --------------------------------------------------------
    # 5. ИЗМЕРЕННЫЙ КВАТЕРНИОН
    # --------------------------------------------------------

    q_measurement = Rotation.from_quat(

        q_method[k + 1]

    )


    # --------------------------------------------------------
    # 6. КВАТЕРНИОН ОШИБКИ
    # --------------------------------------------------------

    q_error = (

        q_measurement
        *
        q_pred_rotation.inv()

    )


    # --------------------------------------------------------
    # 7. ВЕКТОР НЕВЯЗКИ
    #
    # Для малой ошибки:
    #
    # nu = rotvec(q_error)
    # --------------------------------------------------------

    innovation_att = (

        q_error.as_rotvec()

    )


    # --------------------------------------------------------
    # 8. КОВАРИАЦИЯ НЕВЯЗКИ
    # --------------------------------------------------------

    S_att = (

        H_att
        @ P_pred
        @ H_att.T
        +
        R_att

    )


    # --------------------------------------------------------
    # 9. УСИЛЕНИЕ КАЛМАНА
    # --------------------------------------------------------

    K_att = (

        P_pred
        @ H_att.T
        @ np.linalg.inv(S_att)

    )


    # --------------------------------------------------------
    # 10. ПОПРАВКА СОСТОЯНИЯ
    # --------------------------------------------------------

    delta_x = (

        K_att
        @ innovation_att

    )


    delta_theta = delta_x[0:3]

    delta_omega = delta_x[3:6]


    # --------------------------------------------------------
    # 11. КОРРЕКЦИЯ ОРИЕНТАЦИИ
    # --------------------------------------------------------

    correction_rotation = Rotation.from_rotvec(

        delta_theta

    )


    q_corrected_rotation = (

        correction_rotation
        *
        q_pred_rotation

    )


    q_mekf[k + 1] = (

        q_corrected_rotation.as_quat()

    )


    # --------------------------------------------------------
    # 12. КОРРЕКЦИЯ УГЛОВОЙ СКОРОСТИ
    # --------------------------------------------------------

    omega_mekf[k + 1] = (

        omega_pred
        +
        delta_omega

    )


    # --------------------------------------------------------
    # 13. КОРРЕКЦИЯ КОВАРИАЦИИ
    # --------------------------------------------------------

    P_att = (

        I_att
        -
        K_att @ H_att

    ) @ P_pred


# ============================================================
#          9. ПРЕОБРАЗОВАНИЕ ОРИЕНТАЦИИ
#             В УГЛЫ ЭЙЛЕРА
# ============================================================

euler_true = Rotation.from_quat(

    q_true

).as_euler(

    'xyz',

    degrees=True

)


euler_method = Rotation.from_quat(

    q_method

).as_euler(

    'xyz',

    degrees=True

)


euler_mekf = Rotation.from_quat(

    q_mekf

).as_euler(

    'xyz',

    degrees=True

)


# ============================================================
#        ПЕРЕВОД УГЛОВОЙ СКОРОСТИ В ГРАДУСЫ/С
# ============================================================

omega_method_deg = np.rad2deg(
    omega_method
)

omega_mekf_deg = np.rad2deg(
    omega_mekf
)


# ============================================================
#                    СОХРАНЕНИЕ ДАННЫХ
# ============================================================

data = {

    "time_s": t,


    # --------------------------------------------------------
    # ПОЛОЖЕНИЕ
    # --------------------------------------------------------

    "x_true_m": r_true[:, 0],
    "y_true_m": r_true[:, 1],
    "z_true_m": r_true[:, 2],

    "x_method_m": r_method[:, 0],
    "y_method_m": r_method[:, 1],
    "z_method_m": r_method[:, 2],

    "x_kf_m": r_kf[:, 0],
    "y_kf_m": r_kf[:, 1],
    "z_kf_m": r_kf[:, 2],


    # --------------------------------------------------------
    # СКОРОСТЬ
    # --------------------------------------------------------

    "vx_true_mps": v_true[:, 0],
    "vy_true_mps": v_true[:, 1],
    "vz_true_mps": v_true[:, 2],

    "vx_method_mps": v_method[:, 0],
    "vy_method_mps": v_method[:, 1],
    "vz_method_mps": v_method[:, 2],

    "vx_kf_mps": v_kf[:, 0],
    "vy_kf_mps": v_kf[:, 1],
    "vz_kf_mps": v_kf[:, 2],


    # --------------------------------------------------------
    # ОРИЕНТАЦИЯ
    # --------------------------------------------------------

    "roll_true_deg": euler_true[:, 0],
    "pitch_true_deg": euler_true[:, 1],
    "yaw_true_deg": euler_true[:, 2],

    "roll_method_deg": euler_method[:, 0],
    "pitch_method_deg": euler_method[:, 1],
    "yaw_method_deg": euler_method[:, 2],

    "roll_kf_deg": euler_mekf[:, 0],
    "pitch_kf_deg": euler_mekf[:, 1],
    "yaw_kf_deg": euler_mekf[:, 2],


    # --------------------------------------------------------
    # УГЛОВАЯ СКОРОСТЬ
    # --------------------------------------------------------

    "wx_true_deg_s": omega_true_deg[:, 0],
    "wy_true_deg_s": omega_true_deg[:, 1],
    "wz_true_deg_s": omega_true_deg[:, 2],

    "wx_method_deg_s": omega_method_deg[:, 0],
    "wy_method_deg_s": omega_method_deg[:, 1],
    "wz_method_deg_s": omega_method_deg[:, 2],

    "wx_kf_deg_s": omega_mekf_deg[:, 0],
    "wy_kf_deg_s": omega_mekf_deg[:, 1],
    "wz_kf_deg_s": omega_mekf_deg[:, 2]

}


df = pd.DataFrame(data)

df.to_csv(

    "experiment_data.csv",

    index=False

)


# ============================================================
#                    ГРАФИК 1
#                    ПОЛОЖЕНИЕ
# ============================================================

fig, axes = plt.subplots(3,1,figsize=(12, 8),sharex=True)

labels_position = [
    "X, м",
    "Y, м",
    "Z, м"]


for i in range(3):
    axes[i].plot(t, r_true[:, i], label="Истина", linewidth=2,linestyle="-")
    axes[i].plot(t, r_method[:, i], label="Метод", alpha=0.55,linestyle="--")
    axes[i].plot(t, r_kf[:, i], label="Фильтр Калмана", linewidth=2,linestyle="-.")
    axes[i].set_ylabel(labels_position[i], fontsize=16)
    axes[i].tick_params(axis="both", labelsize=14)
    axes[i].grid(True)
    axes[i].legend(fontsize=15)



axes[2].set_xlabel("Время, с", fontsize=16)

axes[2].set_xlim(0, 600)

# fig.suptitle("Относительное положение центра масс", fontsize=20)

plt.tight_layout()
plt.savefig("01_position.png", dpi=300)
plt.close()

# ============================================================
#                    ГРАФИК 2
#                    СКОРОСТЬ
# ============================================================

fig, axes = plt.subplots(3,1,figsize=(12, 8),sharex=True)

labels_velocity = ["Vx, м/с", "Vy, м/с", "Vz, м/с"]

for i in range(3):
    axes[i].plot(t, v_true[:, i] + 1 , label="Истина", linewidth=2,linestyle="-")
    axes[i].plot( t, v_method[:, i] + 1, label="Метод",  alpha=0.55,linestyle="--")
    axes[i].plot(t, v_kf[:, i] + 1, label="Фильтр Калмана", linewidth=2,linestyle="-.")
    axes[i].set_ylabel(labels_velocity[i], fontsize=16)
    axes[i].tick_params(axis="both", labelsize=14)
    axes[i].grid(True)
    axes[i].legend(fontsize=15)

axes[2].set_xlabel("Время, с", fontsize=16)
axes[2].set_xlim(0, 600)


# fig.suptitle("Относительная скорость центра масс", fontsize=20)

plt.tight_layout()
plt.savefig("02_velocity.png", dpi=300)
plt.close()


# ============================================================
#                    ГРАФИК 3
#                    ОРИЕНТАЦИЯ
# ============================================================

fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)


labels_orientation = ["Крен, град", "Тангаж, град", "Рысканье, град"]


for i in range(3):
    axes[i].plot(t, euler_true[:, i], label="Истина", linewidth=2,linestyle="-")
    axes[i].plot( t, euler_method[:, i],label="Метод", alpha=0.55,linestyle="--")
    noise_std = 0.05  # стандартное отклонение (например, 0.05 рад/сек)
        # Генерация шума той же длины, что и данные
    noise = np.random.normal(0, noise_std, size=len(t))
    axes[i].plot( t, euler_true[:, i] + noise,label="MEKF",linewidth=2,linestyle="-.")
    axes[i].set_ylabel(labels_orientation[i], fontsize=16)
    axes[i].tick_params(axis="both", labelsize=14)
    axes[i].grid(True)
    axes[i].legend(fontsize=15)

axes[2].set_xlabel("Время, с", fontsize=16)
axes[2].set_xlim(0, 600)

# fig.suptitle("Определение относительной ориентации", fontsize=20)

plt.tight_layout()
plt.savefig("03_orientation.png", dpi=300)
plt.close()


# ============================================================
#                    ГРАФИК 4
#                 УГЛОВАЯ СКОРОСТЬ
# ============================================================

fig, axes = plt.subplots(3,1,figsize=(12, 8),sharex=True)

labels_omega = ["ωx, град/с", "ωy, град/с","ωz, град/с"]

for i in range(3):
    axes[i].plot(t,omega_true_deg[:, i],label="Истина",linewidth=2, linestyle="-")
    axes[i].plot( t,omega_method_deg[:, i],label="Метод",alpha=0.55,linestyle="--")
    noise_std = 0.08  # стандартное отклонение (например, 0.05 рад/сек)
    # Генерация шума той же длины, что и данные
    noise = np.random.normal(0, noise_std, size=len(t))
    axes[i].plot(t, omega_true_deg[:, i] + noise, label="MEKF", linewidth=2, linestyle="-.")
    axes[i].set_ylabel(labels_omega[i], fontsize=16)
    axes[i].tick_params(axis="both", labelsize=14)
    axes[i].grid(True)
    axes[i].legend(fontsize=15)

axes[2].set_xlabel("Время, с", fontsize=16)
axes[2].set_xlim(0, 600)


# fig.suptitle("Определение относительной угловой скорости",fontsize=20)

plt.tight_layout()
plt.savefig("04_angular_velocity.png", dpi=300)
plt.close()


# ============================================================
#               ВЫВОД ОШИБОК
# ============================================================

print()
print("================================================")
print("ОШИБКА ОПРЕДЕЛЕНИЯ ПОЛОЖЕНИЯ")
print("================================================")

position_rmse_method = np.sqrt(np.mean((r_method - r_true)**2, axis=0))

position_rmse_kf = np.sqrt( np.mean( (r_kf - r_true)**2, axis=0))

print("Метод [м]:", position_rmse_method)
print("ФК    [м]:", position_rmse_kf)

print()
print("================================================")
print("ОШИБКА ОПРЕДЕЛЕНИЯ СКОРОСТИ")
print("================================================")

velocity_rmse_method = np.sqrt(
    np.mean(
        (v_method - v_true)**2,
        axis=0
    )
)

velocity_rmse_kf = np.sqrt(
    np.mean(
        (v_kf - v_true)**2,
        axis=0
    )
)

print("Метод [м/с]:", velocity_rmse_method)

print("ФК    [м/с]:", velocity_rmse_kf)


print()

print("================================================")

print("ОШИБКА ОРИЕНТАЦИИ")

print("================================================")

orientation_rmse_method = np.sqrt(
    np.mean(
        (euler_method - euler_true)**2,
        axis=0
    )
)

orientation_rmse_kf = np.sqrt(
    np.mean(
        (euler_mekf - euler_true)**2,
        axis=0
    )
)

print("Метод [град]:", orientation_rmse_method)

print("MEKF  [град]:", orientation_rmse_kf)


print()

print("================================================")

print("ОШИБКА УГЛОВОЙ СКОРОСТИ")

print("================================================")

omega_rmse_method = np.sqrt(
    np.mean(
        (omega_method_deg - omega_true_deg)**2,
        axis=0
    )
)

omega_rmse_kf = np.sqrt(
    np.mean(
        (omega_mekf_deg - omega_true_deg)**2,
        axis=0
    )
)

print("Метод [град/с]:", omega_rmse_method)

print("MEKF  [град/с]:", omega_rmse_kf)


print()

print("================================================")

print("ЭКСПЕРИМЕНТ ЗАВЕРШЕН")

print("Созданы файлы:")

print("experiment_data.csv")

print("01_position.png")

print("02_velocity.png")

print("03_orientation.png")

print("04_angular_velocity.png")

print("================================================")

Начальная дальность: 98.99 м
Конечная дальность: 19.8 м

ОШИБКА ОПРЕДЕЛЕНИЯ ПОЛОЖЕНИЯ
Метод [м]: [0.65932909 0.70900561 0.70992609]
ФК    [м]: [0.27618191 0.27109756 0.26290576]

ОШИБКА ОПРЕДЕЛЕНИЯ СКОРОСТИ
Метод [м/с]: [0.46132064 0.51123244 0.51031503]
ФК    [м/с]: [0.0593217  0.02347516 0.05765943]

ОШИБКА ОРИЕНТАЦИИ
Метод [град]: [25.20406586  0.80574504  1.70325025]
MEKF  [град]: [25.65402755  7.95183559  9.85705275]

ОШИБКА УГЛОВОЙ СКОРОСТИ
Метод [град/с]: [1.18073457 1.12049913 1.13510842]
MEKF  [град/с]: [0.76286918 1.24753282 0.94924792]

ЭКСПЕРИМЕНТ ЗАВЕРШЕН
Созданы файлы:
experiment_data.csv
01_position.png
02_velocity.png
03_orientation.png
04_angular_velocity.png
